# 텍스트 정제 함수

In [ ]:
!pip install -U langchain-core langchain-community langchain-chroma

In [7]:
# Cell 1: 라이브러리 임포트 및 환경 설정
import json
import os
import re
from langchain_core.documents import Document

# 1. 경로 설정
# rag_v2.ipynb는 rag/ 폴더 안에 있으므로 ../data로 접근합니다.
load_dir = "../data/check_cards_benefits"
db_path = "../data/card_vector_db"

def clean_text(text):
    """
    텍스트 내의 줄바꿈(\n)과 불필요한 연속 공백을 정제합니다.
    """
    if not text:
        return ""
    
    # 1. \n을 공백으로 치환 (단어 결합 방지)
    text = text.replace("\n", " ")
    
    # 2. 여러 개의 공백(스페이스, 탭 등)을 한 칸의 공백으로 축소
    text = re.sub(r'\s+', ' ', text)
    
    # 3. 양 끝의 불필요한 공백 제거
    return text.strip()

# 경로 존재 여부 확인
if os.path.exists(load_dir):
    print(f"✅ 데이터 로드 경로 확인 완료: {load_dir}")
else:
    print(f"❌ 오류: '{load_dir}' 경로를 찾을 수 없습니다. 폴더 위치를 확인해주세요.")

✅ 데이터 로드 경로 확인 완료: ../data/check_cards_benefits


# VectorDB 구축

In [9]:
# Cell 2: 임베딩 모델 로드 및 벡터 DB 구축 (수정본)
import json
import os
import re

# 1. 에러 해결의 핵심: 임포트 경로 수정
from langchain_core.documents import Document 
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 앞서 정의한 텍스트 정제 함수 (원본은 유지하되 DB에만 깨끗하게 저장)
def clean_text(text):
    if not text: return ""
    text = text.replace("\n", " ") # \n을 공백으로 치환
    text = re.sub(r'\s+', ' ', text) # 연속 공백 축소
    return text.strip()

def build_card_vector_db():
    # 경로 설정
    load_dir = "../data/check_cards_benefits"
    db_path = "../data/card_vector_db" 
    
    # 2. 임베딩 모델 로드 (내부적으로 토큰화 수행)
    print("임베딩 모델을 로드 중입니다...")
    embeddings = HuggingFaceEmbeddings(
        model_name="jhgan/ko-sroberta-multitask",
        model_kwargs={'device': 'cpu'}
    )

    documents = []

    if not os.path.exists(load_dir):
        print(f"❌ 오류: 데이터를 찾을 수 없습니다: {load_dir}")
        return

    print(f"'{load_dir}'에서 데이터를 읽어와 정제 중...")
    file_list = [f for f in os.listdir(load_dir) if f.endswith('.json')]

    for file_name in file_list:
        with open(os.path.join(load_dir, file_name), 'r', encoding='utf-8') as f:
            card_data = json.load(f)
        
        # 데이터 정제 적용 (실시간 처리)
        card_name = clean_text(card_data.get('card_name'))
        company = clean_text(card_data.get('company'))
        performance = clean_text(card_data.get('performance'))
        
        for b in card_data.get('benefit', []):
            if b['category'] == "유의사항": #
                continue
                
            # 혜택 내용도 정제
            benefit_category = clean_text(b['category'])
            benefit_content = clean_text(b['content'])
            
            # 검색 성능 최적화를 위한 구조화된 문장 구성
            page_content = (
                f"카드이름: {card_name} | 카드사: {company} | "
                f"혜택 카테고리: {benefit_category} | 상세내용: {benefit_content} | "
                f"전월실적: {performance}"
            )
            
            metadata = {
                "card_name": card_name,
                "company": company,
                "category": benefit_category,
                "performance": performance
            }
            
            documents.append(Document(page_content=page_content, metadata=metadata))

    # 3. 벡터 DB 생성 및 영구 저장
    print(f"총 {len(documents)}개의 정제된 혜택 조각을 벡터화하여 저장합니다...")
    
    vector_db = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory=db_path
    )
    
    # 최신 버전에서는 persist() 호출 없이도 자동 저장되지만 명시적으로 수행
    vector_db.persist()
    print(f"✅ 벡터 DB 구축 완료! 위치: {db_path}")

    # 4. 간단한 테스트 검색
    print("\n[검색 테스트 결과]")
    query = "영화 할인 혜택이 있는 카드 추천해줘"
    results = vector_db.similarity_search_with_score(query, k=3)
    
    for doc, score in results:
        print(f"- [유사도: {score:.4f}] {doc.page_content}")

# 실행
build_card_vector_db()

임베딩 모델을 로드 중입니다...


C:\Users\82104\AppData\Local\Temp\ipykernel_16148\1773119754.py:25: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\82104\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\82104\.cache\huggingface\hub\models--jhgan--ko-sroberta-multitask. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

'../data/check_cards_benefits'에서 데이터를 읽어와 정제 중...
총 1653개의 정제된 혜택 조각을 벡터화하여 저장합니다...


C:\Users\82104\AppData\Local\Temp\ipykernel_16148\1773119754.py:82: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


✅ 벡터 DB 구축 완료! 위치: ../data/card_vector_db

[검색 테스트 결과]
- [유사도: 52.3420] 카드이름: 리디 신한카드 체크 | 카드사: 신한카드 | 혜택 카테고리: 영화 | 상세내용: 영화 예매시 10% 할인(캐시백) | 전월실적: 전월실적 20만원 이상
- [유사도: 53.0024] 카드이름: SOCAR 제휴 SOCAR 신한카드 체크 | 카드사: 신한카드 | 혜택 카테고리: 영화 | 상세내용: 영화 이용 시 10% 할인(캐시백) | 전월실적: 전월실적 20만원 이상
- [유사도: 54.6105] 카드이름: 참! 좋은 kt wiz 카드[체크] | 카드사: IBK기업은행 | 혜택 카테고리: 영화 | 상세내용: 전국 영화관 및 인터넷 예매 4천원 할인 | 전월실적: 전월실적 20만원 이상
